#Montando o Google Drive

In [ ]:
 # Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)
# Monta o Google Drive para acessar arquivos do projeto
try:
  import google.colab
  COLAB = True
  print("Rodando no Colab")
except:
  COLAB = False
  print("Rodando Localmente")

if COLAB:
  from google.colab import drive
  drive.mount('/content/drive', timeout_ms=10200000)


#Importando Bibliotecas

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)
# Bibliotecas do sistema e utilitárias
import os
import random
import time
import platform
import pickle
import cv2


# Bibliotecas científicas
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning e avaliação
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc, classification_report
from sklearn import metrics
from sklearn.preprocessing import label_binarize

# PyTorch e TorchVision
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

from torch.optim import lr_scheduler
from torchvision import datasets, transforms, models as models, utils

# Utilitário de barra de progresso
from tqdm import tqdm

from transformers import AutoProcessor, AutoModelForImageClassification


print("--- Versões do Ambiente de Execução ---")
print(f"Versão do Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"Sistema: {platform.system()} - {platform.release()}")
# Para verificar a versão do CUDA que o PyTorch está usando:
if torch.cuda.is_available():
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA: Não disponível")

#Configurações para reprodutibilidade

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

#Verificando acesso à GPU

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)
#Verifica se a GPU está disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if COLAB:
    !nvidia-smi

#O Dataset

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)
USE_WAVELET_DATASET = False

if COLAB:
  if USE_WAVELET_DATASET:
    ds_path = ""
    print("Usando dataset com pré-processamento Wavelet.")
  else:
    ds_path = ""
    print("Usando dataset original (sem pré-processamento).")
else:
  if USE_WAVELET_DATASET:
    ds_path = ""
    print("Usando dataset com pré-processamento Wavelet.")
  else:
    ds_path = ""
    print("Usando dataset original (sem pré-processamento).")



Definição de Hiperparâmetros

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)

# Tamanho do lote (mini-batch)
# batch size 16 quando for efficientnet para não dar estouro de memória
batch_size = 16

# Taxa de aprendizado
lr = 0.001

# Número de épocas
epochs = 500

Preparando o Conjunto de Dados

Dividindo o dataset em treino, teste e validação

In [ ]:

# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)

# --- ETAPA 1: DIVISÃO DOS ÍNDICES PRIMEIRO ---

# Carrega o dataset bruto (sem transformações)
imagefolder_dataset = datasets.ImageFolder(ds_path)
classes = imagefolder_dataset.classes
total_size = len(imagefolder_dataset)
print(f"Dataset total com {total_size} imagens. Classes: {classes}\n")

# Split dos índices com seed fixa
train_size = int(0.6 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

train_indices, val_indices, test_indices = torch.utils.data.random_split(
    range(total_size),
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

# --- ETAPA 2: CÁLCULO DAS ESTATÍSTICAS APENAS NO CONJUNTO DE TREINO ---

# Dataset temporário com transform mínimo para estatísticas
imagefolder_dataset_for_stats = datasets.ImageFolder(ds_path, transform=transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor()
]))

# Subset só com os índices de treino
train_subset_for_stats = Subset(imagefolder_dataset_for_stats, train_indices)
stats_loader = DataLoader(train_subset_for_stats, batch_size=64, shuffle=False, num_workers=2)

print("Calculando média e desvio padrão APENAS no conjunto de treino...")
mean = 0.
std = 0.
for images, _ in tqdm(stats_loader):
    images = images.view(images.size(0), -1)
    mean += images.mean(1).sum()
    std += images.std(1).sum()

mean /= len(train_subset_for_stats)
std /= len(train_subset_for_stats)
print(f"Média calculada do treino: {mean.item():.5f}")
print(f"Desvio padrão calculado do treino: {std.item():.5f}\n")



In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)

# --- ETAPA 3: CRIAÇÃO DOS DATASETS E DATALOADERS FINAIS ---

# Transforms finais
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean.item()], std=[std.item()])
])

test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[mean.item()], std=[std.item()])
])

# Recria um ImageFolder por transform (garante isolamento total entre os subsets)
train_dataset = Subset(datasets.ImageFolder(ds_path, transform=train_transform), train_indices)
val_dataset = Subset(datasets.ImageFolder(ds_path, transform=test_transform), val_indices)
test_dataset = Subset(datasets.ImageFolder(ds_path, transform=test_transform), test_indices)

# Criação dos dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print("Tamanhos dos conjuntos finais:")
print(f"Treino: {len(train_dataset)}")
print(f"Validação: {len(val_dataset)}")
print(f"Teste: {len(test_dataset)}")


#Seleção do Modelo

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)

def get_model(model_name, num_classes, device):
    """
    Retorna um modelo pré-treinado com a última camada ajustada para num_classes.
    Ajusta a primeira camada convolucional para aceitar 1 canal (grayscale).
    """
    # Carrega o modelo base
    if model_name == 'alexnet':
        model = models.alexnet(weights='IMAGENET1K_V1')
        # Modifica a primeira camada convolucional para aceitar 1 canal
        model.features[0] = nn.Conv2d(1, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    elif model_name == 'resnet50':
        model = models.resnet50(weights='IMAGENET1K_V1')
        # Modifica a primeira camada convolucional para aceitar 1 canal
        model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    elif model_name == 'vgg16':
        model = models.vgg16(weights='IMAGENET1K_V1')
        # Modifica a primeira camada convolucional para aceitar 1 canal
        model.features[0] = nn.Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    elif model_name == 'mobilenet_v2':
        model = models.mobilenet_v2(weights='IMAGENET1K_V1')
        # Modifica a primeira camada convolucional para aceitar 1 canal
        model.features[0][0] = nn.Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    elif model_name == 'efficientnet_b7':
        model = models.efficientnet_b7(weights='IMAGENET1K_V1')
        # Modifica a primeira camada convolucional para aceitar 1 canal
        model.features[0][0] = nn.Conv2d(1, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    elif model_name == 'vit_b_16':
        model = models.vit_b_16(weights='IMAGENET1K_V1')
        # Vision Transformer tem um patch embedding layer em vez de conv. Modificamos ele.
        # O patch embedding layer tem a forma (in_channels, out_channels, kernel_size, kernel_size)
        model.conv_proj = nn.Conv2d(1, model.conv_proj.out_channels, kernel_size=model.conv_proj.kernel_size, stride=model.conv_proj.stride, padding=model.conv_proj.padding)
    else:
        raise ValueError(f"Modelo '{model_name}' não está disponível. Escolha entre: ['alexnet', 'resnet50', 'vgg16', 'mobilenet_v2', 'efficientnet_b7', 'vit_b_16']")

    # Substitui a camada de classificação (classifier head) para a tarefa específica
    if model_name in ['alexnet', 'vgg16']:
        num_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(num_features, num_classes)
    elif model_name == 'resnet50':
        num_features = model.fc.in_features
        model.fc = nn.Linear(num_features, num_classes)
    elif model_name in ['mobilenet_v2', 'efficientnet_b7']:
        num_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(num_features, num_classes)
    elif model_name == 'vit_b_16':
        num_features = model.heads.head.in_features
        model.heads.head = nn.Linear(num_features, num_classes)


    return model.to(device)

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)

model_name = 'vit_b_16'
num_classes = len(classes)
model = get_model(model_name, num_classes, device)

#Definição da função de perda e do otimizador

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

#Outras definições importantes para o treinamento

In [ ]:
# Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)

# Diretório para salvar melhores pesos
if COLAB:
    base_path = ""
else:
    base_path = ""


# Diretório onde os melhores pesos serão salvos
if USE_WAVELET_DATASET:
  save_dir = os.path.join(base_path, "treinamentos", model_name, "wavelet")
  os.makedirs(save_dir, exist_ok=True)
else:
  save_dir = os.path.join(base_path, "treinamentos", model_name, "original")
  os.makedirs(save_dir, exist_ok=True)

best_model_path = os.path.join(save_dir, f"melhor_modelo_{model_name}.pth")
best_weights_path = os.path.join(save_dir, f"melhores_pesos_{model_name}.pth")
checkpoint_path = os.path.join(save_dir, f"checkpoint_{model_name}.pth")


relatorio_dir = os.path.join(save_dir, "relatorios")
os.makedirs(relatorio_dir, exist_ok=True)


print(f"Caminho para salvar o melhor modelo: {best_model_path}")

print(f"Salvando pesos em: {best_weights_path}")

#Treinando o Modelo

In [ ]:
# Inicialização
train_loss_list = []
train_acc_list = []
val_loss_list = []
val_acc_list = []
all_labels = []
all_preds = []
all_probs = []
start_epoch = 0
num_epochs = epochs

# --- INÍCIO DAS VARIÁVEIS DE EARLY STOPPING ---
patience = 10
epochs_no_improve = 0
min_val_loss = float('inf')
# --- FIM DAS VARIÁVEIS DE EARLY STOPPING ---


# Tenta carregar checkpoint (se houver)
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    # Carrega também o min_val_loss para continuar o early stopping corretamente
    if 'min_val_loss' in checkpoint:
        min_val_loss = checkpoint['min_val_loss']
    print(f"Checkpoint carregado (época {start_epoch})")

    metricas_path = os.path.join(relatorio_dir, f"metricas_{model_name}.pkl")
    if os.path.exists(metricas_path):
      with open(metricas_path, "rb") as f:
          metricas = pickle.load(f)
          train_loss_list = metricas.get('train_loss', [])
          val_loss_list = metricas.get('val_loss', [])
          train_acc_list = metricas.get('train_acc', [])
          val_acc_list = metricas.get('val_acc', [])
          all_labels = metricas.get('all_labels', [])
          all_preds = metricas.get('all_preds', [])
          all_probs = metricas.get('all_probs', [])
          print(f" Métricas carregadas de {metricas_path}")

# Inicia temporizador
time_total_start = time.time()

# Loop de épocas
for epoch in range(start_epoch, num_epochs):
    print(f"\nÉpoca {epoch+1}/{num_epochs}")
    model.train()
    running_loss = 0.0
    running_corrects = 0

    # Treinamento
    for inputs, labels in tqdm(train_dataloader, desc="Treinamento"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += torch.sum(preds == labels.data)

    train_loss = running_loss / len(train_dataloader.dataset)
    train_acc = running_corrects.double() / len(train_dataloader.dataset)

    train_loss_list.append(train_loss)
    train_acc_list.append(train_acc.item())

    # Validação
    model.eval()
    val_running_loss = 0.0
    val_running_corrects = 0
    epoch_all_labels = [] # Listas locais para a época atual
    epoch_all_preds = []
    epoch_all_probs = []

    with torch.no_grad():
        for inputs, labels in tqdm(val_dataloader, desc="Validação"):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(outputs, dim=1)
            loss = criterion(outputs, labels)

            epoch_all_labels.extend(labels.cpu().numpy())
            epoch_all_preds.extend(preds.cpu().numpy())
            epoch_all_probs.extend(probs.cpu().numpy())

            val_running_loss += loss.item() * inputs.size(0)
            val_running_corrects += torch.sum(preds == labels.data)

    val_loss = val_running_loss / len(val_dataloader.dataset)
    val_acc = val_running_corrects.double() / len(val_dataloader.dataset)

    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc.item())

    print(f"Treino: Loss {train_loss:.4f} | Acc {train_acc:.4f}")
    print(f"Validação: Loss {val_loss:.4f} | Acc {val_acc:.4f}")

    # Salva checkpoint da época atual
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
        'min_val_loss': min_val_loss # Salva o min_val_loss no checkpoint
    }, checkpoint_path)
    print(f"Checkpoint da época {epoch + 1} salvo em {checkpoint_path}")

    # Define e salva as listas de métricas
    metricas = {
      'train_loss': train_loss_list, 'val_loss': val_loss_list,
      'train_acc': train_acc_list, 'val_acc': val_acc_list,
      'all_labels': epoch_all_labels, 'all_preds': epoch_all_preds,
      'all_probs': epoch_all_probs
    }
    with open(os.path.join(relatorio_dir, f'metricas_{model_name}_training.pkl'), 'wb') as f:
      pickle.dump(metricas, f)

    # --- INÍCIO DA LÓGICA DE EARLY STOPPING E SALVAMENTO DO MELHOR MODELO ---
    if val_loss < min_val_loss:
        print(f"Validação Loss melhorou ({min_val_loss:.4f} --> {val_loss:.4f}). Salvando melhor modelo...")
        min_val_loss = val_loss
        epochs_no_improve = 0
        # Salva o modelo considerado o melhor até agora
        torch.save(model.state_dict(), best_weights_path)
    else:
        epochs_no_improve += 1
        print(f"Validação Loss não melhorou. Contador: {epochs_no_improve}/{patience}")

    if epochs_no_improve >= patience:
        print(f"\nEARLY STOPPING! O modelo não melhora há {patience} épocas.")
        break  # Interrompe o loop de treinamento
    # --- FIM DA LÓGICA DE EARLY STOPPING ---

# Tempo total
time_total = time.time() - time_total_start
print(f"\nTreinamento finalizado em {int(time_total // 60)}m {int(time_total % 60)}s")

#Avaliação do Modelo (conjuntos de treinamento e validaçao)

In [ ]:
# Para plotar as métricas sem ter que treinar o modelo novamente. Para que isso funcione, rode todas as células onde existe o comentário:
# " Necessário rodar quando se precisa fazer quelquer coisa, menos treinamento (também quando faz o treinamento)"

# Rodar essa célula não é necessário quando se acabou de sair do treinamento

metricas_path = os.path.join(relatorio_dir, f"metricas_{model_name}_training.pkl")

train_loss_list = []
val_loss_list = []
train_acc_list = []
val_acc_list = []
all_labels = []
all_preds = []
all_probs = []

if os.path.exists(metricas_path):
    with open(metricas_path, "rb") as f:
        metricas = pickle.load(f)
        train_loss_list = metricas.get('train_loss', [])
        val_loss_list = metricas.get('val_loss', [])
        train_acc_list = metricas.get('train_acc', [])
        val_acc_list = metricas.get('val_acc', [])
        all_labels = metricas.get('all_labels', [])
        all_preds = metricas.get('all_preds', [])
        all_probs = metricas.get('all_probs', [])
    print(f"Métricas carregadas de {metricas_path}")

In [ ]:
epochs_list = list(range(1, len(train_loss_list) + 1))
loss_title = f'Loss over {len(epochs_list)} Epochs'
acc_title = f'Accuracy over {len(epochs_list)} Epochs'

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss
ax1.set_title(loss_title)
ax1.plot(epochs_list, train_loss_list, c='magenta', ls='--', label='Train Loss')
ax1.plot(epochs_list, val_loss_list, c='green', ls='--', label='Val. Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend(loc='best')
ax1.grid(True)

# Accuracy
ax2.set_title(acc_title)
ax2.plot(epochs_list, train_acc_list, c='magenta', ls='-', label='Train Accuracy')
ax2.plot(epochs_list, val_acc_list, c='green', ls='-', label='Val. Accuracy')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend(loc='best')
ax2.grid(True)

# Salvar gráfico
plot_path = os.path.join(relatorio_dir, "grafico_loss_accuracy.png")
plt.tight_layout()
plt.savefig(plot_path)
plt.close()
print(f"Gráfico de perda e acurácia salvo em: {plot_path}")

In [ ]:
relatorio_txt_path = os.path.join(relatorio_dir, "relatorio_classificacao_validacao.txt")
with open(relatorio_txt_path, "w") as f:
    f.write("Relatório de Classificação - Validação\n\n")
    f.write(classification_report(all_labels, all_preds, target_names=classes))
print(f"Relatório salvo em: {relatorio_txt_path}")

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão (Validação)")
plt.grid(False)

matriz_path = os.path.join(relatorio_dir, "matriz_confusao_validacao.png")
plt.savefig(matriz_path)
plt.close()
print(f"Matriz de confusão salva em: {matriz_path}")

In [ ]:
#CURVA ROC

min_len = min(len(all_labels), len(all_probs))
all_labels = all_labels[:min_len]
all_probs_np = np.array(all_probs[:min_len])

# Verifica se o problema é binário e se as probabilidades têm duas colunas
if num_classes == 2 and all_probs_np.shape[1] == 2:
    probs_pos = all_probs_np[:, 1]
    fpr, tpr, _ = roc_curve(all_labels, probs_pos)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('Curva ROC (Binária - Validação)')
    plt.legend(loc="lower right")
    plt.grid(True)

else:
    # Multiclasse ou formato diferente — usa macroaverage
    true_binarized = label_binarize(all_labels, classes=list(range(num_classes)))
    fpr, tpr, _ = roc_curve(true_binarized.ravel(), all_probs_np.ravel())
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('Curva ROC (MIcro-Average - Validação)')
    plt.legend(loc="lower right")
    plt.grid(True)

# Salvar imagem
roc_validacao_path = os.path.join(relatorio_dir, "curva_roc_validacao.png")
plt.savefig(roc_validacao_path)
plt.close()
print(f"Curva ROC da validação salva em: {roc_validacao_path}")

#Avaliação Final do Modelo

#Reinicialização



Restaurar modelo e carergar os pesos

À partir deste ponto é possível rodar mesmo sem ter os dados do treinamento do modelo. Para que isso funcione, rode todas as células onde existe o comentário:
**" Necessário rodar quando se precisa fazer qualquer coisa, menos treinamento (também quando faz o treinamento)"**

*Mesmo que você esteja rodando o trecho abaixo com os dados do treinamento em memória, é necessário executar todas as células abaixo*

In [ ]:
# Número de classes (ajuste conforme o seu dataset)
num_classes = len(classes)

# Nome do modelo usado durante o treinamento
model_name = 'vit_b_16'

# Dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instancia o modelo com a arquitetura correta e última camada adaptada
model = get_model(model_name, num_classes, device)

# Carrega os melhores pesos
model.load_state_dict(torch.load(best_weights_path, map_location=device))
model.to(device)
model.eval()

print(f"Modelo '{model_name}' restaurado com sucesso usando os melhores pesos.")


#Código GradCAM

In [ ]:
class GradCAM:
    def __init__(self, modelo, camada_alvo):
        self.modelo = modelo
        self.camada_alvo = camada_alvo
        self.gradiente = None
        self.ativacao = None

        self.camada_alvo.register_forward_hook(self.salvar_ativacao)
        self.camada_alvo.register_backward_hook(self.salvar_gradiente)

    def salvar_ativacao(self, module, input, output):
        self.ativacao = output

    def salvar_gradiente(self, module, grad_input, grad_output):
        self.gradiente = grad_output[0]

    def get_cam(self, tensor_entrada, classe_alvo=None):
        self.modelo.eval()
        saida = self.modelo(tensor_entrada)

        if classe_alvo is None:
            classe_alvo = torch.argmax(saida, dim=1)

        self.modelo.zero_grad()
        one_hot = torch.zeros_like(saida)
        one_hot[0][classe_alvo] = 1

        saida.backward(gradient=one_hot, retain_graph=True)

        if isinstance(self.camada_alvo, nn.LayerNorm):
            patch_h = int(tensor_entrada.shape[-2] / self.modelo.conv_proj.kernel_size[0])
            patch_w = int(tensor_entrada.shape[-1] / self.modelo.conv_proj.kernel_size[1])


            gradiente_spatial = self.gradiente[:, 1:].view(-1, patch_h, patch_w, self.gradiente.shape[-1])
            ativacao_spatial = self.ativacao[:, 1:].view(-1, patch_h, patch_w, self.ativacao.shape[-1])

            gradiente_spatial = gradiente_spatial.permute(0, 3, 1, 2)
            ativacao_spatial = ativacao_spatial.permute(0, 3, 1, 2)

            pesos = torch.mean(gradiente_spatial, dim=(2, 3), keepdim=True)
            cam = torch.sum(pesos * ativacao_spatial, dim=1, keepdim=True)

        elif isinstance(self.camada_alvo, (nn.Conv2d, nn.Sequential)): # Para as CNNs
            pesos = torch.mean(self.gradiente, dim=(2, 3), keepdim=True)
            cam = torch.sum(pesos * self.ativacao, dim=1, keepdim=True)
        elif isinstance(self.camada_alvo, models.resnet.Bottleneck):
             target_conv_layer = self.camada_alvo.conv3
             pesos = torch.mean(self.gradiente, dim=(2, 3), keepdim=True)
             cam = torch.sum(pesos * self.ativacao, dim=1, keepdim=True)
        else:
             raise TypeError(f"Unsupported target layer type for GradCAM: {type(self.camada_alvo)}")


        cam = torch.relu(cam)

        # normalizacao
        epsilon = 1e-8
        cam = cam - torch.min(cam)
        cam = cam / (torch.max(cam) - torch.min(cam) + epsilon)

        return cam.detach().cpu()

In [ ]:
heatmap_dir = os.path.join(save_dir, "heatmaps")
os.makedirs(heatmap_dir, exist_ok=True)

print(f"Salvando Heatmaps em {heatmap_dir}")

if model_name == 'alexnet':
    camada_alvo = model.features[-3]
elif model_name == 'vgg16':
    camada_alvo = model.features[-3]
elif model_name == 'resnet50':
    camada_alvo = model.layer4[-1]

elif model_name == 'mobilenet_v2':
    camada_alvo = model.features[-1]
elif model_name == 'efficientnet_b7':
    camada_alvo = model.features[-1]
elif model_name == 'vit_b_16':
    camada_alvo = model.encoder.layers[-1].ln_1

gradcam = GradCAM(model, camada_alvo)

print(f"GradCAM inicializado para o modelo '{model_name}' na camada: {camada_alvo.__class__.__name__}")

In [ ]:
#Função para visualizar a imagem original, o heatmap, a imagem sobreposta e as classes previstas e reais


def visualizar_heatmaps(imagem, heatmap, dataset_mean, dataset_std, classe_prevista=None, classe_real=None, alpha=0.5, caminho_saida="heatmaps", nome_arquivo="imagem"):

    # 1. Converter tensor para numpy e DESNORMALIZAR CORRETAMENTE (para 1 canal)
    imagem_np = imagem.squeeze().cpu().detach().numpy() # Garante que está na CPU e detacha do grafo de computação
    # Desnormaliza usando a média e std do seu dataset
    imagem_np = (imagem_np * dataset_std) + dataset_mean
    imagem_np = np.clip(imagem_np, 0, 1)
    imagem_uint8_gray = np.uint8(255 * imagem_np)

    # 2. Converter a imagem de escala de cinza para BGR (3 canais) para a visualização
    imagem_bgr = cv2.cvtColor(imagem_uint8_gray, cv2.COLOR_GRAY2BGR)


    # Heatmap puro - Redimensionar o heatmap para o tamanho original da imagem BGR
    # Adicionar verificação se heatmap é um tensor antes de chamar squeeze().numpy()
    if isinstance(heatmap, torch.Tensor):
        heatmap_np = heatmap.squeeze().numpy()
    else:
        print(f"Aviso: Heatmap não e um tensor, nao e possivel gerar a visualizacao para {nome_arquivo}.")
        heatmap_np = np.zeros((imagem_bgr.shape[0], imagem_bgr.shape[1]), dtype=np.float32)


    heatmap_resized = cv2.resize(heatmap_np, (imagem_bgr.shape[1], imagem_bgr.shape[0]), interpolation=cv2.INTER_LINEAR)
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)


    # Overlay (agora ambas as imagens estão em BGR, as cores serão corretas)
    overlay_bgr = cv2.addWeighted(imagem_bgr, alpha, heatmap_bgr, 1 - alpha, 0)

    # Combinar horizontalmente (todas já estão em BGR)
    imagem_final = cv2.hconcat([imagem_bgr, heatmap_bgr, overlay_bgr])

    status = None
    if classe_real is not None and classe_prevista is not None:
        status = "CORRECT" if classe_real == classe_prevista else "INCORRECT"

    texto = ""
    if classe_real is not None:
        texto += f"True: {classe_real} "
    if classe_prevista is not None:
        texto += f"| Predicted: {classe_prevista} "
    if status:
        texto += f"| {status}"

    if texto:
        cor_texto = (255, 255, 255)
        cv2.putText(imagem_final, texto, (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, cor_texto, 2, cv2.LINE_AA)

    cor_borda = (0, 255, 0) if status == "CORRECT" else (0, 0, 255)

    espessura_borda = 10
    imagem_borda = cv2.copyMakeBorder(
        imagem_final, top=espessura_borda, bottom=espessura_borda,
        left=espessura_borda, right=espessura_borda,
        borderType=cv2.BORDER_CONSTANT, value=cor_borda
    )

    os.makedirs(caminho_saida, exist_ok=True)
    caminho_completo = os.path.join(caminho_saida, f"{nome_arquivo}.png")
    cv2.imwrite(caminho_completo, imagem_borda)

    return overlay_bgr

In [ ]:
def desativa_inplace_relu(model):
    for nome, modulo in list(model.named_modules()):
        if isinstance(modulo, nn.ReLU) and modulo.inplace:
            pai = model
            nome_split = nome.split('.')
            for subnome in nome_split[:-1]:
                pai = getattr(pai, subnome)
            setattr(pai, nome_split[-1], nn.ReLU(inplace=False))


desativa_inplace_relu(model)

#Avaliação no conjunto de teste

In [ ]:
# --- ETAPA 2: AVALIAÇÃO E GERAÇÃO DE GRADCAMS ---

model.eval()

test_running_loss = 0.0
test_correct = 0
all_labels = []
all_preds = []
all_probs = []

with torch.no_grad():
    for inputs, labels in tqdm(test_dataloader, desc="Avaliando o modelo"):
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)

        probs = torch.softmax(outputs, dim=1)

        preds = torch.argmax(outputs, dim=1)
        loss = criterion(outputs, labels)

        test_running_loss += loss.item() * inputs.size(0)
        test_correct += (preds == labels).sum().item()

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

test_loss = test_running_loss / len(test_dataloader.dataset)
test_acc = test_correct / len(test_dataloader.dataset)
print(f"\nResultados do Teste: Loss {test_loss:.4f} | Acc {test_acc:.4f}")


metricas_teste = {
    'test_loss': test_loss,
    'test_acc': test_acc,
    'all_labels': all_labels,
    'all_preds': all_preds,
    'all_probs': all_probs
}

metricas_teste_path = os.path.join(relatorio_dir, f'metricas_{model_name}_test.pkl')
with open(metricas_teste_path, 'wb') as f:
    pickle.dump(metricas_teste, f)
print(f"Métricas de teste salvas em: {metricas_teste_path}")


print("\nGerando heatmaps para algumas imagens de exemplo...")
heatmap_dir = os.path.join(save_dir, "heatmaps")
os.makedirs(heatmap_dir, exist_ok=True)



for i, (inputs, labels) in enumerate(test_dataloader):

    inputs, labels = inputs.to(device), labels.to(device)

    for j in range(inputs.size(0)):
        imagem_tensor = inputs[j].unsqueeze(0)
        label_real = labels[j].item()

        outputs = model(imagem_tensor)
        classe_prevista_idx = torch.argmax(outputs, dim=1).item()

        heatmap_tensor = gradcam.get_cam(imagem_tensor)

        visualizar_heatmaps(
            imagem=imagem_tensor.squeeze(0).cpu(),
            heatmap=heatmap_tensor,
            dataset_mean=mean.item(),
            dataset_std=std.item(),
            classe_real=classes[label_real],
            classe_prevista=classes[classe_prevista_idx],
            caminho_saida=heatmap_dir,
            nome_arquivo=f"heatmap_{i}_{j}"
        )

print(f"Heatmaps salvos em {heatmap_dir}")

#Métricas Finais (Avaliação no conjunto de testes)

In [ ]:
# --- ETAPA 3: GERAÇÃO DE RELATÓRIOS A PARTIR DOS DADOS SALVOS ---

metricas_teste_path = os.path.join(relatorio_dir, f'metricas_{model_name}_test.pkl')
if os.path.exists(metricas_teste_path):
    with open(metricas_teste_path, 'rb') as f:
        metricas_teste = pickle.load(f)

    all_labels = metricas_teste['all_labels']
    all_preds = metricas_teste['all_preds']
    all_probs = metricas_teste['all_probs']
    print(f"\nMétricas de teste carregadas de {metricas_teste_path} para geração de relatórios.")

else:
    print("ERRO: Arquivo de métricas de teste não encontrado. Rode a avaliação primeiro.")


# Relatório de Classificação
relatorio_teste_txt_path = os.path.join(relatorio_dir, "relatorio_classificacao_teste.txt")
with open(relatorio_teste_txt_path, "w") as f:
    f.write("Relatório de Classificação - Teste\n\n")
    f.write(classification_report(all_labels, all_preds, target_names=classes))
print(f"Relatório de teste salvo em: {relatorio_teste_txt_path}")

#Matriz de Confusão
cm_test = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test, display_labels=classes)
disp.plot(cmap=plt.cm.Blues)
plt.title("Matriz de Confusão (Teste)")
plt.grid(False)

matriz_teste_path = os.path.join(relatorio_dir, "matriz_confusao_teste.png")
plt.savefig(matriz_teste_path)
plt.close()
print(f"Matriz de confusão do teste salva em: {matriz_teste_path}")


min_len = min(len(all_labels), len(all_probs))
all_labels = all_labels[:min_len]
all_probs_np = np.array(all_probs[:min_len])


#Curva ROC
# Verifica se o problema é binário ou multiclasse
if num_classes == 2 and all_probs_np.shape[1] == 2:
    probs_pos = all_probs_np[:, 1]
    fpr, tpr, _ = roc_curve(all_labels, probs_pos)
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('Curva ROC (Binária - Teste)')
    plt.legend(loc="lower right")
    plt.grid(True)

else:
    # Multiclasse — calcula macro-average
    true_binarized = label_binarize(all_labels, classes=list(range(num_classes)))
    fpr, tpr, _ = roc_curve(true_binarized.ravel(), all_probs_np.ravel())
    roc_auc = auc(fpr, tpr)

    plt.figure()
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'AUC = {roc_auc:.2f}')
    plt.plot([0, 1], [0, 1], 'r--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('Curva ROC (Micro-Average - Teste)')
    plt.legend(loc="lower right")
    plt.grid(True)

roc_teste_path = os.path.join(relatorio_dir, "curva_roc_teste.png")
plt.savefig(roc_teste_path)
plt.close()
print(f"Curva ROC do teste salva em: {roc_teste_path}")